# Model Comparison

Three models, one test set, and the honest question: which one is actually better?

We put logistic regression, XGBoost, and DeBERTa side by side on AUC, F1, calibration, and a McNemar's test to check whether the performance difference between the top two is statistically real or just noise. The winner carries extra weight going into the stress-test.

In [ ]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import (
    roc_auc_score, f1_score,
    RocCurveDisplay, PrecisionRecallDisplay, CalibrationDisplay
)
from inference_lens.utils.logging import setup_logging

setup_logging()
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

import os
os.makedirs('../../reports/figures', exist_ok=True)

## Load test predictions

Logistic regression and XGBoost predictions come from notebook 01. DeBERTa predictions come from the Colab notebook. If you haven't run the DeBERTa notebook yet, the comparison will show only two models -- that's fine, add DeBERTa when the checkpoint is ready.

In [ ]:
preds_df = pd.read_parquet('../../data/processed/test_predictions.parquet')

deberta_available = False
deberta_path = '../../data/processed/deberta_test_predictions.parquet'

if os.path.exists(deberta_path):
    deberta_df = pd.read_parquet(deberta_path)
    preds_df['deberta_prob'] = deberta_df['deberta_prob'].values
    preds_df['deberta_pred'] = deberta_df['deberta_pred'].values
    deberta_available = True
    print('DeBERTa predictions loaded.')
else:
    print('DeBERTa predictions not found -- run notebook 02 on Colab first.')
    print('Continuing with LogReg and XGBoost only.')

print(f'Test set: {len(preds_df):,} samples')
preds_df.head(3)

## Summary metrics table

In [ ]:
y_true = preds_df['y_true'].values

models = {
    'Logistic Regression': ('logreg_prob', 'logreg_pred'),
    'XGBoost':             ('xgb_prob',    'xgb_pred'),
}
if deberta_available:
    models['DeBERTa-v3-small'] = ('deberta_prob', 'deberta_pred')

rows = []
for name, (prob_col, pred_col) in models.items():
    probs = preds_df[prob_col].values
    preds = preds_df[pred_col].values
    rows.append({
        'Model': name,
        'AUC-ROC': round(roc_auc_score(y_true, probs), 4),
        'F1 (macro)': round(f1_score(y_true, preds, average='macro'), 4),
        'F1 (chosen)': round(f1_score(y_true, preds, pos_label=1), 4),
        'Accuracy': round((preds == y_true).mean(), 4),
    })

summary = pd.DataFrame(rows).set_index('Model')
summary

## ROC curves

In [ ]:
colors = ['steelblue', 'darkorange', 'mediumseagreen']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for (name, (prob_col, pred_col)), color in zip(models.items(), colors):
    RocCurveDisplay.from_predictions(
        y_true, preds_df[prob_col].values,
        name=name, ax=axes[0], color=color
    )
    PrecisionRecallDisplay.from_predictions(
        y_true, preds_df[prob_col].values,
        name=name, ax=axes[1], color=color
    )

axes[0].set_title('ROC curves')
axes[1].set_title('Precision-Recall curves')
plt.suptitle('All models on HH-RLHF test set', fontsize=13)
plt.tight_layout()
plt.savefig('../../reports/figures/18_all_models_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()

## Calibration comparison

This matters more than it gets credit for. A model that says 0.9 confidence should be right 90% of the time. If it's not, its scores can't be trusted in the stress-test phase.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for (name, (prob_col, _)), color in zip(models.items(), colors):
    CalibrationDisplay.from_predictions(
        y_true, preds_df[prob_col].values,
        n_bins=10, name=name, ax=ax, color=color
    )

ax.set_title('Calibration curves (reliability diagrams)')
plt.tight_layout()
plt.savefig('../../reports/figures/19_all_models_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## McNemar's test

McNemar's test checks whether two classifiers make different kinds of mistakes. If model A gets a sample right while model B gets it wrong, and vice versa, enough times -- that's a statistically significant difference. A low p-value means the difference in performance is real, not sampling noise.

In [ ]:
model_names = list(models.keys())
pred_cols = [v[1] for v in models.values()]

print("McNemar's test results (p-value < 0.05 means the difference is statistically significant):")
print()

for i in range(len(model_names)):
    for j in range(i+1, len(model_names)):
        preds_a = preds_df[pred_cols[i]].values
        preds_b = preds_df[pred_cols[j]].values

        # contingency table: both correct, a-only, b-only, both wrong
        a_right = (preds_a == y_true)
        b_right = (preds_b == y_true)

        n_a_not_b = ((a_right) & (~b_right)).sum()
        n_b_not_a = ((~a_right) & (b_right)).sum()

        table = np.array([[((a_right) & (b_right)).sum(), n_a_not_b],
                           [n_b_not_a, ((~a_right) & (~b_right)).sum()]])

        result = mcnemar(table, exact=False, correction=True)
        sig = 'SIGNIFICANT' if result.pvalue < 0.05 else 'not significant'
        print(f'  {model_names[i]} vs {model_names[j]}')
        print(f'    statistic: {result.statistic:.4f}  p-value: {result.pvalue:.4f}  ({sig})')
        print()

## Per-class error analysis

Which model makes the most mistakes on chosen responses vs rejected ones? A model that confidently misclassifies chosen responses is particularly problematic for our use case.

In [ ]:
print(f'Error breakdown (% of each class misclassified):')
print(f'{"Model":<25} {"Chosen errors":>15} {"Rejected errors":>17}')
print('-' * 60)

for name, (_, pred_col) in models.items():
    preds = preds_df[pred_col].values
    chosen_err   = (preds[y_true == 1] != y_true[y_true == 1]).mean() * 100
    rejected_err = (preds[y_true == 0] != y_true[y_true == 0]).mean() * 100
    print(f'{name:<25} {chosen_err:>14.1f}%  {rejected_err:>15.1f}%')

## Save summary for the report

In [ ]:
summary.to_csv('../../reports/model_comparison_summary.csv')
print('Saved reports/model_comparison_summary.csv')
print()
print('Final rankings:')
print(summary.sort_values('AUC-ROC', ascending=False).to_string())